# NB07: RAST Validation

**Purpose**: Validate evidence channels against RAST annotations as ground truth.

**Validation levels**:
1. **EC-level**: Which ECs does each channel cover? Do they agree with RAST's EC set?
2. **Protein-level**: For shared proteins (Tier 1 vs RAST), compute precision/recall/F1
   on protein-EC pairs.
3. **Reaction-level**: Coverage of balanced reactions.

**Limitation**: Only Tier 1 (UniProt accessions) shares ID space with RAST (UniProt proteins).
Tier 2 uses gene_cluster IDs and Tier 3 uses locus IDs — cross-entity validation
is limited to EC-level agreement.

**Memory note**: RAST has 32M+ pairs; tier1/tier2 have millions of rows.
This notebook avoids holding multiple large DataFrames and their set copies
simultaneously — each tier is loaded, its EC set extracted, then freed.

**Output**: Validation statistics, per-channel F1 scores

In [1]:
import pandas as pd
import numpy as np
import gc

DATA_DIR = '../data'

ec_bridge = pd.read_parquet(f'{DATA_DIR}/ec_to_reaction.parquet')
bridge_ecs = set(ec_bridge['ec'])

balanced_ids = set(
    pd.read_csv(f'{DATA_DIR}/reactions_all.tsv', sep='\t', usecols=['id', 'status'])
    .query("status == 'OK'")['id']
    .str.replace('seed.reaction:', '', regex=False)
)

rast = pd.read_parquet(f'{DATA_DIR}/rast_protein_ec.parquet')
rast_ecs = set(rast['ec'])
rast_proteins = set(rast['protein_id'])

print(f'RAST ground truth:')
print(f'  Protein-EC pairs: {len(rast):,}')
print(f'  Unique proteins:  {len(rast_proteins):,}')
print(f'  Unique ECs:       {len(rast_ecs):,}')
print(f'  ECs in bridge:    {len(rast_ecs & bridge_ecs):,}')
print(f'  Balanced rxns:    {len(balanced_ids):,}')

RAST ground truth:
  Protein-EC pairs: 32,420,974
  Unique proteins:  30,363,667
  Unique ECs:       2,439
  ECs in bridge:    2,135
  Balanced rxns:    34,343


## 1. EC-Level Validation

For each channel, compare its set of ECs against RAST's EC set.

In [2]:
def ec_stats(name, ec_set):
    tp = len(ec_set & rast_ecs)
    prec = tp / len(ec_set) if ec_set else 0
    recall = tp / len(rast_ecs) if rast_ecs else 0
    f1 = 2 * prec * recall / (prec + recall) if (prec + recall) > 0 else 0
    return {'channel': name, 'n_ecs': len(ec_set), 'tp': tp,
            'prec': prec, 'recall': recall, 'f1': f1}

ec_results = []

# --- Tier 1: load, extract EC sets, keep DataFrame for protein-level validation later ---
tier1 = pd.read_parquet(f'{DATA_DIR}/uniprot_native_protein_ec.parquet')
tier1_ec_sets = {
    'UniProt EC': set(tier1[tier1['channels'].str.contains('uniprot_ec', na=False)]['ec']),
    'BRENDA': set(tier1[tier1['channels'].str.contains('brenda', na=False)]['ec']),
    'Rhea': set(tier1[tier1['channels'].str.contains('rhea', na=False)]['ec']),
}
tier1_all_ecs = set(tier1['ec'])
for name, ec_set in tier1_ec_sets.items():
    ec_results.append(ec_stats(name, ec_set))
ec_results.append(ec_stats('Tier 1 combined', tier1_all_ecs))
del tier1_ec_sets

# --- Tier 2: load, extract EC sets only, then free ---
tier2 = pd.read_parquet(f'{DATA_DIR}/pangenome_gc_ec.parquet')
tier2_ec_sets = {
    'eggNOG EC': set(tier2[tier2['channels'].str.contains('eggnog_ec', na=False)]['ec']),
    'bakta EC': set(tier2[tier2['channels'].str.contains('bakta_ec', na=False)]['ec']),
}
tier2_all_ecs = set(tier2['ec'])
for name, ec_set in tier2_ec_sets.items():
    ec_results.append(ec_stats(name, ec_set))
ec_results.append(ec_stats('Tier 2 combined', tier2_all_ecs))
del tier2, tier2_ec_sets
gc.collect()

# --- Tier 3: load, extract EC sets only, then free ---
tier3 = pd.read_parquet(f'{DATA_DIR}/curated_evidence_ec.parquet')
tier3_ec_sets = {
    'PaperBLAST': set(tier3[tier3['channel'] == 'paperblast']['ec']),
    'seedclass': set(tier3[tier3['channel'] == 'seedclass']['ec']),
    'besthitmetacyc': set(tier3[tier3['channel'] == 'besthitmetacyc_ec']['ec']),
}
tier3_all_ecs = set(tier3['ec'])
for name, ec_set in tier3_ec_sets.items():
    ec_results.append(ec_stats(name, ec_set))
ec_results.append(ec_stats('Tier 3 combined', tier3_all_ecs))
del tier3, tier3_ec_sets
gc.collect()

# --- All tiers combined ---
all_ecs = tier1_all_ecs | tier2_all_ecs | tier3_all_ecs
ec_results.append(ec_stats('ALL COMBINED', all_ecs))

print(f'{"Channel":<25s} {"ECs":>6s} {"∩RAST":>6s} {"Prec":>6s} {"Recall":>6s} {"F1":>6s}')
print('-' * 57)
for r in ec_results:
    print(f'{r["channel"]:<25s} {r["n_ecs"]:>6,} {r["tp"]:>6,} {r["prec"]:>5.1%} {r["recall"]:>5.1%} {r["f1"]:>5.3f}')

Channel                      ECs  ∩RAST   Prec Recall     F1
---------------------------------------------------------
UniProt EC                 4,938  2,132 43.2% 87.4% 0.578
BRENDA                     3,941  1,817 46.1% 74.5% 0.570
Rhea                       3,907  1,825 46.7% 74.8% 0.575
Tier 1 combined            5,032  2,132 42.4% 87.4% 0.571
eggNOG EC                  3,286  1,929 58.7% 79.1% 0.674
bakta EC                   2,926  1,870 63.9% 76.7% 0.697
Tier 2 combined            3,783  2,059 54.4% 84.4% 0.662
PaperBLAST                 4,700  2,085 44.4% 85.5% 0.584
seedclass                  1,260  1,215 96.4% 49.8% 0.657
besthitmetacyc             1,677  1,223 72.9% 50.1% 0.594
Tier 3 combined            4,780  2,111 44.2% 86.6% 0.585
ALL COMBINED               5,123  2,134 41.7% 87.5% 0.564


## 2. Protein-Level Validation (Tier 1 vs RAST)

Both Tier 1 and RAST use UniProt accessions. For proteins present in both,
compute per-protein precision/recall on EC assignments.

In [3]:
tier1_proteins = set(tier1['protein'])
shared_proteins = tier1_proteins & rast_proteins

print(f'Protein ID overlap:')
print(f'  Tier 1 proteins:  {len(tier1_proteins):,}')
print(f'  RAST proteins:    {len(rast_proteins):,}')
print(f'  Shared:           {len(shared_proteins):,}')
print(f'  Tier 1 only:      {len(tier1_proteins - rast_proteins):,}')
print(f'  RAST only:        {len(rast_proteins - tier1_proteins):,}')

del tier1_proteins

Protein ID overlap:
  Tier 1 proteins:  26,549,024
  RAST proteins:    30,363,667
  Shared:           15,952,112
  Tier 1 only:      10,596,912


  RAST only:        14,411,555


In [4]:
t1_shared = tier1[tier1['protein'].isin(shared_proteins)][['protein', 'ec']].drop_duplicates()
rast_shared = rast[rast['protein_id'].isin(shared_proteins)][['protein_id', 'ec']].drop_duplicates()
rast_shared = rast_shared.rename(columns={'protein_id': 'protein'})

merged = t1_shared.merge(rast_shared, on=['protein', 'ec'], how='outer', indicator=True)
tp = int((merged['_merge'] == 'both').sum())
fp = int((merged['_merge'] == 'left_only').sum())
fn = int((merged['_merge'] == 'right_only').sum())

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'Protein-EC pair validation (shared proteins):')
print(f'  Tier 1 pairs: {len(t1_shared):,}')
print(f'  RAST pairs:   {len(rast_shared):,}')
print(f'  TP (both):    {tp:,}')
print(f'  FP (T1 only): {fp:,}')
print(f'  FN (RAST only): {fn:,}')
print(f'\n  Precision: {precision:.4f}')
print(f'  Recall:    {recall:.4f}')
print(f'  F1:        {f1:.4f}')

tier1_combined_f1 = f1
tier1_combined_prec = precision
tier1_combined_rec = recall

del merged
gc.collect()

Protein-EC pair validation (shared proteins):
  Tier 1 pairs: 16,700,160
  RAST pairs:   17,102,353
  TP (both):    14,149,439
  FP (T1 only): 2,550,721
  FN (RAST only): 2,952,914

  Precision: 0.8473
  Recall:    0.8273
  F1:        0.8372


16

In [5]:
print(f'Per-channel protein-level F1 (on shared proteins):')
print(f'{"Channel":<20s} {"Pairs":>10s} {"TP":>8s} {"FP":>8s} {"Prec":>7s} {"Recall":>7s} {"F1":>7s}')
print('-' * 70)

per_channel_results = []
for ch_name, ch_label in [('uniprot_ec', 'UniProt EC'), ('brenda', 'BRENDA'), ('rhea', 'Rhea')]:
    ch_df = tier1[tier1['channels'].str.contains(ch_name, na=False)]
    ch_shared = ch_df[ch_df['protein'].isin(shared_proteins)][['protein', 'ec']].drop_duplicates()
    m = ch_shared.merge(rast_shared, on=['protein', 'ec'], how='outer', indicator=True)
    tp = int((m['_merge'] == 'both').sum())
    fp = int((m['_merge'] == 'left_only').sum())
    fn = int((m['_merge'] == 'right_only').sum())
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    print(f'{ch_label:<20s} {len(ch_shared):>10,} {tp:>8,} {fp:>8,} {prec:>6.1%} {rec:>6.1%} {f1:>6.3f}')
    per_channel_results.append({'channel': ch_label, 'pairs': len(ch_shared),
                                'tp': tp, 'fp': fp, 'prec': prec, 'recall': rec, 'f1': f1})
    del ch_df, ch_shared, m

del tier1, t1_shared
gc.collect()

Per-channel protein-level F1 (on shared proteins):
Channel                   Pairs       TP       FP    Prec  Recall      F1
----------------------------------------------------------------------


UniProt EC           16,696,813 14,147,756 2,549,057  84.7%  82.7%  0.837


BRENDA                   14,067   10,864    3,203  77.2%   0.1%  0.001


Rhea                    157,123  144,974   12,149  92.3%   0.8%  0.017


22

## 3. Reaction-Level Validation

Compare reaction sets reachable by each channel vs RAST.

In [6]:
rast_rxns = set(ec_bridge[ec_bridge['ec'].isin(rast_ecs)]['rxn_bare'])

ec_set_map = {
    'Tier 1 combined': tier1_all_ecs,
    'Tier 2 combined': tier2_all_ecs,
    'Tier 3 combined': tier3_all_ecs,
    'ALL COMBINED': all_ecs,
}

print(f'{"Channel":<25s} {"Rxns":>8s} {"∩RAST":>8s} {"Prec":>7s} {"Recall":>7s} {"F1":>7s}')
print('-' * 62)

for name, ec_set in ec_set_map.items():
    ch_rxns = set(ec_bridge[ec_bridge['ec'].isin(ec_set)]['rxn_bare'])
    tp = len(ch_rxns & rast_rxns)
    prec = tp / len(ch_rxns) if ch_rxns else 0
    recall = tp / len(rast_rxns) if rast_rxns else 0
    f1 = 2 * prec * recall / (prec + recall) if (prec + recall) > 0 else 0
    print(f'{name:<25s} {len(ch_rxns):>8,} {tp:>8,} {prec:>6.1%} {recall:>6.1%} {f1:>6.3f}')

combined_rxns = set(ec_bridge[ec_bridge['ec'].isin(all_ecs)]['rxn_bare'])
combined_rxn_coverage = len(combined_rxns) / len(balanced_ids)

Channel                       Rxns    ∩RAST    Prec  Recall      F1
--------------------------------------------------------------
Tier 1 combined             17,215   10,725  62.3% 100.0%  0.768
Tier 2 combined             14,557   10,566  72.6%  98.5%  0.836
Tier 3 combined             16,254   10,687  65.7%  99.6%  0.792
ALL COMBINED                17,319   10,729  61.9% 100.0%  0.765


## 4. Error Analysis

Examine FP/FN patterns to understand disagreements.

In [7]:
tier1_only_ecs = tier1_all_ecs - rast_ecs
rast_only_ecs = rast_ecs - tier1_all_ecs

print(f'EC-level disagreements (Tier 1 vs RAST):')
print(f'  Tier 1 ECs not in RAST: {len(tier1_only_ecs):,}')
print(f'  RAST ECs not in Tier 1: {len(rast_only_ecs):,}')

tier1_only_rxns = set(ec_bridge[ec_bridge['ec'].isin(tier1_only_ecs)]['rxn_bare'])
rast_only_rxns = set(ec_bridge[ec_bridge['ec'].isin(rast_only_ecs)]['rxn_bare'])

print(f'\n  Reactions reachable only via Tier 1 ECs: {len(tier1_only_rxns):,}')
print(f'  Reactions reachable only via RAST ECs:   {len(rast_only_rxns):,}')

print(f'\nRAST-only ECs (first 20):')
rast_only_sorted = sorted(rast_only_ecs)
for ec in rast_only_sorted[:20]:
    n_proteins = int(rast[rast['ec'] == ec]['protein_id'].nunique())
    print(f'  {ec:<15s}  ({n_proteins:,} proteins in RAST)')

EC-level disagreements (Tier 1 vs RAST):
  Tier 1 ECs not in RAST: 2,900
  RAST ECs not in Tier 1: 307

  Reactions reachable only via Tier 1 ECs: 7,528
  Reactions reachable only via RAST ECs:   7

RAST-only ECs (first 20):


  1.1.2.6          (139 proteins in RAST)
  1.1.98.2         (2,827 proteins in RAST)


  1.1.99.20        (395 proteins in RAST)
  1.1.99.22        (759 proteins in RAST)


  1.1.99.4         (2,141 proteins in RAST)
  1.11.1.9         (38,279 proteins in RAST)


  1.11.2.4         (1,211 proteins in RAST)
  1.12.98.1        (661 proteins in RAST)


...

  1.13.11.24       (4,132 proteins in RAST)
  1.13.11.49       (60 proteins in RAST)


  1.13.12.16       (1,863 proteins in RAST)
  1.14.1.-         (1,004 proteins in RAST)


  1.14.12.20       (148 proteins in RAST)
  1.14.12.4        (60 proteins in RAST)


  1.14.13.221      (594 proteins in RAST)
  1.14.15.12       (134 proteins in RAST)


  1.14.19.2        (7,759 proteins in RAST)
  1.14.99.29       (3,854 proteins in RAST)


In [8]:
tier1_reload = pd.read_parquet(f'{DATA_DIR}/uniprot_native_protein_ec.parquet',
                               columns=['protein', 'ec'])
t1_sh = tier1_reload[tier1_reload['protein'].isin(shared_proteins)][['protein', 'ec']].drop_duplicates()
del tier1_reload
gc.collect()

merged_fp = t1_sh.merge(rast_shared, on=['protein', 'ec'], how='outer', indicator=True)

fp_df = merged_fp[merged_fp['_merge'] == 'left_only'][['protein', 'ec']]
fn_df = merged_fp[merged_fp['_merge'] == 'right_only'][['protein', 'ec']]

print(f'False positive analysis (Tier 1 says EC, RAST disagrees):')
print(f'  Total FP pairs: {len(fp_df):,}')
print(f'  Top FP ECs (most frequent):')
fp_ec_counts = fp_df['ec'].value_counts().head(10)
for ec, ct in fp_ec_counts.items():
    print(f'    {ec:<15s}  {ct:>6,} proteins')

print(f'\nFalse negative analysis (RAST says EC, Tier 1 misses):')
print(f'  Total FN pairs: {len(fn_df):,}')
print(f'  Top FN ECs (most frequent):')
fn_ec_counts = fn_df['ec'].value_counts().head(10)
for ec, ct in fn_ec_counts.items():
    print(f'    {ec:<15s}  {ct:>6,} proteins')

del t1_sh, merged_fp, fp_df, fn_df
gc.collect()

False positive analysis (Tier 1 says EC, RAST disagrees):
  Total FP pairs: 2,550,721
  Top FP ECs (most frequent):
    7.1.1.2          160,611 proteins
    7.1.2.2          101,345 proteins
    5.4.99.-         92,321 proteins
    2.6.1.-          61,599 proteins
    2.3.1.-          58,516 proteins
    2.1.1.-          54,244 proteins
    2.1.3.15         48,678 proteins
    6.3.5.-          44,270 proteins
    2.5.1.-          43,303 proteins
    1.2.1.-          43,302 proteins

False negative analysis (RAST says EC, Tier 1 misses):
  Total FN pairs: 2,952,914
  Top FN ECs (most frequent):
    1.6.5.3          176,741 proteins
    3.6.3.14         101,582 proteins
    6.3.5.6          67,409 proteins
    6.4.1.2          50,423 proteins
    6.3.5.7          44,321 proteins
    2.6.1.1          43,424 proteins
    1.6.1.2          40,045 proteins
    2.7.3.-          36,444 proteins
    2.3.1.16         34,316 proteins
    1.2.1.12         31,555 proteins


0

## 5. Hypothesis Testing

Test H0/H1 from the research plan:
- **H0**: No combination achieves >50% coverage with F1 >0.7
- **H1**: Multi-evidence achieves >70% coverage with F1 >0.8

In [9]:
print('=' * 60)
print('HYPOTHESIS TEST RESULTS')
print('=' * 60)
print(f'\nCombined reaction coverage: {len(combined_rxns):,} / {len(balanced_ids):,} ({100*combined_rxn_coverage:.1f}%)')
print(f'Protein-level F1 (Tier 1 vs RAST on shared proteins): {tier1_combined_f1:.4f}')
print(f'  Precision: {tier1_combined_prec:.4f}')
print(f'  Recall:    {tier1_combined_rec:.4f}')
print()

h0_coverage = combined_rxn_coverage > 0.50
h0_f1 = tier1_combined_f1 > 0.70

h1_coverage = combined_rxn_coverage > 0.70
h1_f1 = tier1_combined_f1 > 0.80

print(f'H0 test (coverage >50% AND F1 >0.7):')
print(f'  Coverage >50%: {"YES" if h0_coverage else "NO"} ({100*combined_rxn_coverage:.1f}%)')
print(f'  F1 >0.7:       {"YES" if h0_f1 else "NO"} ({tier1_combined_f1:.4f})')
print(f'  H0 rejected:   {"YES" if h0_coverage and h0_f1 else "NO"}')
print()
print(f'H1 test (coverage >70% AND F1 >0.8):')
print(f'  Coverage >70%: {"YES" if h1_coverage else "NO"} ({100*combined_rxn_coverage:.1f}%)')
print(f'  F1 >0.8:       {"YES" if h1_f1 else "NO"} ({tier1_combined_f1:.4f})')
print(f'  H1 supported:  {"YES" if h1_coverage and h1_f1 else "NO"}')
print()
if h0_coverage and h0_f1 and not (h1_coverage and h1_f1):
    print('OUTCOME: H0 rejected but H1 not fully supported.')
    print('Multi-evidence mapping achieves meaningful coverage with good accuracy,')
    print('but does not reach the ambitious H1 thresholds.')
elif h1_coverage and h1_f1:
    print('OUTCOME: H1 fully supported. Multi-evidence mapping achieves')
    print('high coverage with high accuracy.')
else:
    print('OUTCOME: H0 cannot be rejected. Coverage and/or accuracy')
    print('remain below the 50%/0.7 thresholds.')

HYPOTHESIS TEST RESULTS

Combined reaction coverage: 17,319 / 34,343 (50.4%)
Protein-level F1 (Tier 1 vs RAST on shared proteins): 0.8372
  Precision: 0.8473
  Recall:    0.8273

H0 test (coverage >50% AND F1 >0.7):
  Coverage >50%: YES (50.4%)
  F1 >0.7:       YES (0.8372)
  H0 rejected:   YES

H1 test (coverage >70% AND F1 >0.8):
  Coverage >70%: NO (50.4%)
  F1 >0.8:       YES (0.8372)
  H1 supported:  NO

OUTCOME: H0 rejected but H1 not fully supported.
Multi-evidence mapping achieves meaningful coverage with good accuracy,
but does not reach the ambitious H1 thresholds.


## 6. Summary

In [10]:
print('=' * 60)
print('NB07 RAST VALIDATION SUMMARY')
print('=' * 60)
print(f'\nProtein-level F1 (Tier 1 vs RAST, {len(shared_proteins):,} shared proteins):')
print(f'  Precision: {tier1_combined_prec:.4f}')
print(f'  Recall:    {tier1_combined_rec:.4f}')
print(f'  F1:        {tier1_combined_f1:.4f}')
print(f'\nCombined reaction coverage: {100*combined_rxn_coverage:.1f}%')
print(f'  ({len(combined_rxns):,} / {len(balanced_ids):,} balanced reactions)')
print(f'\nEC-level results saved above.')
print(f'\nNext: NB08 -- summary visualizations')

del rast, rast_shared, shared_proteins, rast_proteins
del ec_bridge, bridge_ecs, balanced_ids
del tier1_all_ecs, tier2_all_ecs, tier3_all_ecs, all_ecs
gc.collect()

NB07 RAST VALIDATION SUMMARY

Protein-level F1 (Tier 1 vs RAST, 15,952,112 shared proteins):
  Precision: 0.8473
  Recall:    0.8273
  F1:        0.8372

Combined reaction coverage: 50.4%
  (17,319 / 34,343 balanced reactions)

EC-level results saved above.

Next: NB08 -- summary visualizations


0